# Textbooks PDF Vectorization to Lakebase
Reads every PDF in `/Volumes/quickstart_catalog/quickstart_schema/textbooks`, extracts text with PyMuPDF, splits it into overlapping chunks, embeds each chunk with the `databricks-gte-large-en` foundation model (1024 dims), and loads the results into Lakebase project `shared`, database `textbooks`, table `public.textbook_chunks` (pgvector, HNSW cosine index).

Re-runnable: rows for each processed file are replaced on every run.

In [0]:
%pip install -q --upgrade "databricks-sdk>=0.118.0" pymupdf "psycopg[binary]"
dbutils.library.restartPython()

In [0]:
VOLUME_PATH = "/Volumes/quickstart_catalog/quickstart_schema/textbooks/"
LAKEBASE_ENDPOINT = "projects/shared/branches/production/endpoints/primary"
DATABASE = "textbooks"
TABLE = "public.textbook_chunks"
EMBED_MODEL = "databricks-gte-large-en"  # 1024-dim embeddings
EMBED_DIM = 1024
CHUNK_CHARS = 1800
OVERLAP_CHARS = 200
EMBED_BATCH = 64
EMBED_THREADS = 8

In [0]:
import psycopg
from databricks.sdk import WorkspaceClient

w = WorkspaceClient()
HOST = w.postgres.get_endpoint(name=LAKEBASE_ENDPOINT).status.hosts.host
USER = w.current_user.me().user_name

def get_conn():
    """Open a Postgres connection using a short-lived OAuth token (valid 1 hour)."""
    token = w.postgres.generate_database_credential(endpoint=LAKEBASE_ENDPOINT).token
    return psycopg.connect(host=HOST, dbname=DATABASE, user=USER, password=token,
                           sslmode="require", autocommit=True)

print("Lakebase host:", HOST)

In [0]:
import re
import pandas as pd
import pymupdf

pymupdf.TOOLS.mupdf_display_errors(False)  # silence harmless annotation warnings

def chunk_text(text, source_file, page_start, page_end, rows):
    """Split text into ~CHUNK_CHARS windows on whitespace, with OVERLAP_CHARS overlap."""
    pos = 0
    while pos < len(text):
        end = min(pos + CHUNK_CHARS, len(text))
        if end < len(text):
            space = text.rfind(" ", pos + CHUNK_CHARS - 300, end)
            if space > pos:
                end = space
        chunk = text[pos:end].strip().replace("\x00", "")
        if len(chunk) > 50:
            rows.append(dict(source_file=source_file, page_start=page_start,
                             page_end=page_end, content=chunk))
        if end >= len(text):
            break
        pos = max(end - OVERLAP_CHARS, pos + 1)

pdfs = sorted(f.name for f in dbutils.fs.ls(VOLUME_PATH) if f.name.lower().endswith(".pdf"))
rows = []
for fn in pdfs:
    doc = pymupdf.open(VOLUME_PATH + fn)
    buf, start_page = "", None
    for i in range(doc.page_count):
        txt = re.sub(r"[ \t]+", " ", doc[i].get_text()).strip()
        if not txt:
            continue
        if start_page is None:
            start_page = i + 1
        buf += ("\n" if buf else "") + txt
        # group a few pages together so chunks keep a page range
        if len(buf) >= CHUNK_CHARS * 3:
            chunk_text(buf, fn, start_page, i + 1, rows)
            buf, start_page = "", None
    if buf:
        chunk_text(buf, fn, start_page, doc.page_count, rows)
    print(f"{fn}: {doc.page_count} pages")
    doc.close()

chunks = pd.DataFrame(rows)
chunks["chunk_index"] = chunks.groupby("source_file").cumcount()
display(chunks.groupby("source_file").size().reset_index(name="chunks"))

In [0]:
import time
from concurrent.futures import ThreadPoolExecutor

def embed(texts):
    """Call the GTE foundation model, retrying with backoff on transient errors."""
    for attempt in range(5):
        try:
            resp = w.serving_endpoints.query(name=EMBED_MODEL, input=list(texts))
            return [d.embedding for d in sorted(resp.data, key=lambda d: d.index)]
        except Exception as e:
            last_err = e
            time.sleep(2 ** attempt)
    raise last_err

records = chunks.to_dict("records")
batches = [records[i:i + EMBED_BATCH] for i in range(0, len(records), EMBED_BATCH)]
with ThreadPoolExecutor(EMBED_THREADS) as ex:
    batch_embeddings = list(ex.map(lambda b: embed([r["content"] for r in b]), batches))

for b, embs in zip(batches, batch_embeddings):
    for r, e in zip(b, embs):
        assert len(e) == EMBED_DIM
        r["embedding"] = e
print(f"Embedded {len(records)} chunks")

In [0]:
with get_conn() as conn:
    conn.execute("CREATE EXTENSION IF NOT EXISTS vector")
    conn.execute(f"""
        CREATE TABLE IF NOT EXISTS {TABLE} (
            id bigserial PRIMARY KEY,
            source_file text NOT NULL,
            chunk_index int NOT NULL,
            page_start int,
            page_end int,
            content text NOT NULL,
            embedding vector({EMBED_DIM}) NOT NULL,
            created_at timestamptz DEFAULT now(),
            UNIQUE (source_file, chunk_index)
        )""")
    with conn.transaction(), conn.cursor() as cur:
        # replace all rows for the files processed in this run
        cur.execute(f"DELETE FROM {TABLE} WHERE source_file = ANY(%s)", (pdfs,))
        cur.executemany(
            f"""INSERT INTO {TABLE} (source_file, chunk_index, page_start, page_end, content, embedding)
                VALUES (%s, %s, %s, %s, %s, %s::vector)""",
            [(r["source_file"], int(r["chunk_index"]), int(r["page_start"]), int(r["page_end"]),
              r["content"], str(r["embedding"])) for r in records])
    conn.execute(f"CREATE INDEX IF NOT EXISTS textbook_chunks_embedding_hnsw "
                 f"ON {TABLE} USING hnsw (embedding vector_cosine_ops)")
print("Load complete")

In [0]:
with get_conn() as conn:
    summary = conn.execute(f"""
        SELECT source_file, count(*) AS chunks, min(vector_dims(embedding)) AS dims
        FROM {TABLE} GROUP BY 1 ORDER BY 1""").fetchall()
display(pd.DataFrame(summary, columns=["source_file", "chunks", "dims"]))
assert sum(s[1] for s in summary if s[0] in pdfs) == len(records)

In [0]:
question = "What is the difference between FIFO and LIFO inventory methods?"
q_vec = str(embed([question])[0])
with get_conn() as conn:
    hits = conn.execute(f"""
        SELECT source_file, page_start, page_end,
               round((1 - (embedding <=> %s::vector))::numeric, 3) AS similarity,
               left(content, 200) AS snippet
        FROM {TABLE}
        ORDER BY embedding <=> %s::vector
        LIMIT 5""", (q_vec, q_vec)).fetchall()
display(pd.DataFrame(hits, columns=["source_file", "page_start", "page_end", "similarity", "snippet"]))